# GeoStat_py · Workbench principal autosuficiente e iterativo (Colab)

Flujo recomendado:
1. Bootstrap autosuficiente.
2. Upload CSV local y carga con `service`.
3. Autodetección + configuración X/Y/Z/target.
4. Editar parámetros analíticos.
5. Ejecutar **recalculate_analysis()** para refrescar EDA + variografía inline.


In [ ]:
# 0) Bootstrap autosuficiente (sesión fresca de Colab)
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/joelmanrique91-lgtm/GeoStat_py.git"
BRANCH = "main"
BASE_DIR = "/content"
REPO_DIR_NAME = "GeoStat_py"
REPO_DIR = Path(BASE_DIR) / REPO_DIR_NAME

if not REPO_DIR.exists():
    print(f"Repo no encontrado en {REPO_DIR}. Clonando...")
    clone_cmd = ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)]
    clone_result = subprocess.run(clone_cmd, text=True, capture_output=True, check=False)
    print(clone_result.stdout)
    if clone_result.returncode != 0:
        print(clone_result.stderr)
        raise RuntimeError(f"No se pudo clonar el repo (code={clone_result.returncode}).")

bootstrap_module_path = str(REPO_DIR / "colab")
if bootstrap_module_path in sys.path:
    sys.path.remove(bootstrap_module_path)
sys.path.insert(0, bootstrap_module_path)

from bootstrap import clone_or_update_repo, configure_sys_path, create_service, install_requirements, validate_imports

repo_root = clone_or_update_repo(REPO_URL, REPO_DIR, BRANCH)
install_requirements(Path(repo_root) / "colab" / "requirements_colab.txt")
configure_sys_path(repo_root)

imports_to_check = [
    "app.services.geostat_service",
    "app.services.visualization_service",
    "app.models.dataset_model",
]
results = validate_imports(imports_to_check)
if any(not r.ok for r in results):
    for r in results:
        print(r)
    raise RuntimeError("Falló validación de imports.")

service = create_service()
print("Service listo:", type(service).__name__)


In [ ]:
# 1) Imports analíticos y estado base
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from app.services.visualization_service import compute_experimental_variogram

plt.style.use("seaborn-v0_8-whitegrid")
UPLOAD_DIR = Path("/content/geostat_uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

csv_path = None
dataset_loaded = False
config_applied = False
x_col = y_col = z_col = target_col = domain_col = hole_id_col = ""

def _safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


In [ ]:
# 2) Upload CSV local (memoria del usuario)
from google.colab import files

uploaded = files.upload()
if uploaded:
    uploaded_name = list(uploaded.keys())[0]
    csv_path = UPLOAD_DIR / uploaded_name
    csv_path.write_bytes(uploaded[uploaded_name])
    print("CSV subido:", uploaded_name)
    print("Ruta:", csv_path)
else:
    print("No se subió CSV todavía. Puedes re-ejecutar esta celda cuando quieras.")


In [ ]:
# 3) Cargar dataset + autodetección real del servicio
columns = []
autodetected = {}

if csv_path is None:
    print("Carga omitida: falta CSV.")
else:
    load_result = service.load_csv(str(csv_path))
    print("load_csv.success:", load_result.success)
    print("load_csv.message:", load_result.message)
    dataset_loaded = bool(load_result.success)

    if dataset_loaded:
        columns = service.get_available_columns()
        autodetected = service.get_autodetected_columns()
        print("Columnas:", columns)
        print("Autodetección:")
        print(json.dumps(autodetected, ensure_ascii=False, indent=2))


In [ ]:
# 4) Configuración X/Y/Z/target (editable)
X_COLUMN_OVERRIDE = ""
Y_COLUMN_OVERRIDE = ""
Z_COLUMN_OVERRIDE = ""
TARGET_COLUMN_OVERRIDE = ""
DOMAIN_COLUMN_OVERRIDE = ""
HOLE_ID_COLUMN_OVERRIDE = ""

if not dataset_loaded:
    print("Configuración omitida: primero carga dataset.")
else:
    x_col = (X_COLUMN_OVERRIDE or autodetected.get("x") or "").strip()
    y_col = (Y_COLUMN_OVERRIDE or autodetected.get("y") or "").strip()
    z_col = (Z_COLUMN_OVERRIDE or autodetected.get("z") or "").strip()
    target_col = (TARGET_COLUMN_OVERRIDE or autodetected.get("target") or "").strip()
    domain_col = (DOMAIN_COLUMN_OVERRIDE or autodetected.get("domain") or "").strip()
    hole_id_col = (HOLE_ID_COLUMN_OVERRIDE or autodetected.get("hole_id") or "").strip()

    if not x_col or not y_col or not target_col:
        print("No se pudo resolver X/Y/target. Ajusta overrides y re-ejecuta.")
    else:
        if not z_col:
            synthetic_z_col = "__z_colab__"
            df_tmp = service.current_dataset.dataframe.copy()
            df_tmp[synthetic_z_col] = 0.0
            synthetic_csv_path = UPLOAD_DIR / f"{Path(csv_path).stem}__with_z.csv"
            df_tmp.to_csv(synthetic_csv_path, index=False)
            reload_result = service.load_csv(str(synthetic_csv_path))
            if not reload_result.success:
                raise RuntimeError(f"No se pudo crear Z sintética: {reload_result.message}")
            z_col = synthetic_z_col
            print(f"Z vacía: se usará columna sintética '{synthetic_z_col}'.")

        cfg_result = service.set_variable_config(
            x_column=x_col,
            y_column=y_col,
            z_column=z_col,
            target_column=target_col,
            hole_id_column=hole_id_col or None,
            domain_column=domain_col or None,
        )
        print("set_variable_config.success:", cfg_result.success)
        print("set_variable_config.message:", cfg_result.message)
        config_applied = bool(cfg_result.success)


In [ ]:
# 5) Parámetros analíticos centralizados (editar aquí y luego recalcular)
ANALYTICS_PARAMS = {
    "target_override": "",              # opcional
    "domain_filter_column": "",         # opcional (ej. domain_col)
    "domain_filter_value": "",          # opcional
    "hole_id_filter_column": "",        # opcional (ej. hole_id_col)
    "hole_id_filter_value": "",         # opcional
    "drop_missing_target": True,
    "target_min": None,                  # opcional float
    "target_max": None,                  # opcional float
    "row_limit": None,                   # opcional int para exploración visual

    "eda_bins": 30,
    "eda_show_probability": True,
    "eda_show_boxplot": True,
    "eda_show_domain_view": True,

    "vario_lag_distance": 10.0,
    "vario_n_lags": 12,
    "vario_lag_tolerance": 5.0,
    "vario_max_distance": 120.0,
    "vario_azimuth": 0.0,
    "vario_dip": 0.0,
    "vario_ang_tol_h": 90.0,
    "vario_ang_tol_v": 90.0,
    "vario_band_width": 0.0,
    "vario_band_height": 0.0,
    "vario_estimator": "classical",
}
ANALYTICS_PARAMS


In [ ]:
# 6) Función única de recálculo (EDA + variografía)
def recalculate_analysis(params: dict | None = None) -> dict:
    params = params or ANALYTICS_PARAMS
    result = {"ok": False, "message": "", "rows_after_filters": 0}

    if not dataset_loaded or not config_applied or service.current_dataset is None:
        msg = "Dataset/configuración no listos. Ejecuta upload/carga/configuración primero."
        print(msg)
        result["message"] = msg
        return result

    df = service.current_dataset.dataframe.copy()
    active_target = str(params.get("target_override") or target_col).strip() or target_col
    if active_target not in df.columns:
        msg = f"Target '{active_target}' no existe en dataset."
        print(msg)
        result["message"] = msg
        return result

    # Filtros notebook-only (sin tocar repo principal)
    if params.get("drop_missing_target", True):
        df = df[df[active_target].notna()]

    domain_filter_column = str(params.get("domain_filter_column") or "").strip()
    domain_filter_value = str(params.get("domain_filter_value") or "").strip()
    if domain_filter_column and domain_filter_value and domain_filter_column in df.columns:
        df = df[df[domain_filter_column].astype(str) == domain_filter_value]

    hole_filter_column = str(params.get("hole_id_filter_column") or "").strip()
    hole_filter_value = str(params.get("hole_id_filter_value") or "").strip()
    if hole_filter_column and hole_filter_value and hole_filter_column in df.columns:
        df = df[df[hole_filter_column].astype(str) == hole_filter_value]

    target_num = _safe_numeric(df[active_target])
    tmin = params.get("target_min")
    tmax = params.get("target_max")
    if tmin is not None:
        df = df[target_num >= float(tmin)]
        target_num = _safe_numeric(df[active_target])
    if tmax is not None:
        df = df[target_num <= float(tmax)]
        target_num = _safe_numeric(df[active_target])

    row_limit = params.get("row_limit")
    if row_limit is not None:
        df = df.head(int(row_limit))

    if df.empty:
        msg = "El dataset quedó vacío después de aplicar filtros."
        print(msg)
        result["message"] = msg
        return result

    result["rows_after_filters"] = int(len(df))

    # Resumen tabular
    target_num = _safe_numeric(df[active_target]).dropna()
    summary = {
        "rows": len(df),
        "columns": len(df.columns),
        "target": active_target,
        "target_valid": int(target_num.shape[0]),
        "target_mean": float(target_num.mean()) if not target_num.empty else math.nan,
        "target_std": float(target_num.std()) if target_num.shape[0] > 1 else math.nan,
        "target_min": float(target_num.min()) if not target_num.empty else math.nan,
        "target_max": float(target_num.max()) if not target_num.empty else math.nan,
    }
    display(pd.DataFrame(summary.items(), columns=["metric", "value"]))

    # EDA inline
    bins = int(params.get("eda_bins", 30))
    show_prob = bool(params.get("eda_show_probability", True))
    show_box = bool(params.get("eda_show_boxplot", True))
    show_domain = bool(params.get("eda_show_domain_view", True))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    vals = target_num.to_numpy(dtype=float)
    if vals.size:
        axes[0].hist(vals, bins=max(5, bins), color="#1f77b4", alpha=0.85)
        axes[0].set_title("Histograma")
    else:
        axes[0].text(0.5, 0.5, "Sin datos para histograma", ha="center", va="center")

    if show_box and vals.size:
        axes[1].boxplot(vals, vert=True)
        axes[1].set_title("Boxplot")
    else:
        axes[1].text(0.5, 0.5, "Boxplot desactivado/no disponible", ha="center", va="center")

    if show_prob and vals.size >= 3:
        sorted_vals = np.sort(vals)
        n = len(sorted_vals)
        probs = (np.arange(n) + 0.5) / n
        theoretical = np.array([float(np.sqrt(2) * 0.0) for _ in probs])
        # Aproximación simple sin SciPy para máxima robustez en entorno notebook:
        theoretical = np.linspace(-2.5, 2.5, n)
        axes[2].scatter(theoretical, sorted_vals, s=10)
        axes[2].set_title("Probability plot (aprox)")
    else:
        axes[2].text(0.5, 0.5, "Probability desactivado/no disponible", ha="center", va="center")

    plt.tight_layout()
    plt.show()

    if show_domain:
        dcol = domain_filter_column or (domain_col if domain_col in df.columns else "")
        if dcol and dcol in df.columns:
            grouped = (
                df[[dcol, active_target]]
                .assign(_target_num=_safe_numeric(df[active_target]))
                .dropna(subset=["_target_num"]) 
                .groupby(dcol)["_target_num"]
            )
            top = grouped.mean().sort_values(ascending=False).head(12)
            if not top.empty:
                plt.figure(figsize=(10, 3.5))
                top.plot(kind="bar", color="#6c8ebf")
                plt.title(f"Media de target por dominio ({dcol})")
                plt.tight_layout()
                plt.show()
            else:
                print("Vista por dominio: sin datos numéricos suficientes.")
        else:
            print("Vista por dominio omitida: no hay columna de dominio usable.")

    # Variografía inline sobre dataframe filtrado
    try:
        vario = compute_experimental_variogram(
            df,
            x_col=x_col,
            y_col=y_col,
            z_col=z_col,
            target_col=active_target,
            lag=float(params.get("vario_lag_distance", 10.0)),
            n_lags=int(params.get("vario_n_lags", 12)),
            max_distance=float(params.get("vario_max_distance", 120.0)),
            lag_tolerance=float(params.get("vario_lag_tolerance", 5.0)),
            azimuth=float(params.get("vario_azimuth", 0.0)),
            dip=float(params.get("vario_dip", 0.0)),
            ang_tol_h=float(params.get("vario_ang_tol_h", 90.0)),
            ang_tol_v=float(params.get("vario_ang_tol_v", 90.0)),
            band_width=float(params.get("vario_band_width", 0.0)),
            band_height=float(params.get("vario_band_height", 0.0)),
            max_points=2500,
        )

        table = pd.DataFrame({
            "lag_center": vario.lag_centers,
            "gamma": vario.gamma_values,
            "npairs": vario.pair_counts,
        })
        display(table)

        fig, ax1 = plt.subplots(figsize=(8, 4))
        ax1.plot(vario.lag_centers, vario.gamma_values, marker="o", color="#2ca02c")
        ax1.set_xlabel("Lag center")
        ax1.set_ylabel("Semivarianza")
        ax1.set_title("Variograma experimental")

        ax2 = ax1.twinx()
        width = max(1.0, (max(vario.lag_centers) / max(1, len(vario.lag_centers))) * 0.6)
        ax2.bar(vario.lag_centers, vario.pair_counts, alpha=0.18, width=width, color="#ff7f0e")
        ax2.set_ylabel("N pares")

        plt.tight_layout()
        plt.show()
    except Exception as exc:  # noqa: BLE001
        print(f"Variografía no ejecutable con filtros/parámetros actuales: {exc}")

    result["ok"] = True
    result["message"] = "Recalculo completado"
    return result


In [ ]:
# 7) Ejecución inicial / re-ejecución manual
# Edita ANALYTICS_PARAMS y vuelve a correr esta celda para refrescar todo.
analysis_state = recalculate_analysis(ANALYTICS_PARAMS)
analysis_state


## Re-ejecución iterativa
- Cambia valores en `ANALYTICS_PARAMS`.
- Re-ejecuta la celda de `recalculate_analysis(...)`.
- Se actualizan tabla resumen, EDA y variografía inline.

> No depende de widgets; funciona por celdas de forma estable.
